# Linear Algebra for Machine Learning

Distilled from Géron's [math_linear_algebra.ipynb](https://github.com/ageron/handson-ml3/blob/main/math_linear_algebra.ipynb). Only the parts ML actually uses. Proofs and decompositions deferred until a specific topic forces them.

**Goal:** after this notebook you should be able to read `w · x + b`, know every shape involved, and debug shape mismatches on your own.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Vectors

A vector is an ordered list of numbers. In ML it represents **one observation** (input) or **one prediction** (output).

- **MNIST input:** 784 numbers, one per pixel → `x ∈ ℝ⁷⁸⁴`
- **Binary prediction ("is it a 5?"):** 1 number → `ℝ`
- **10-class prediction:** 10 probabilities → `ℝ¹⁰`

Dimension `n` = number of features. Geometrically: a point (or arrow from origin) in n-dim space.

In [ ]:
x = np.array([10.5, 5.2, 3.25, 7.0])
print('x =', x)
print('shape:', x.shape)    # (4,) — 1D array with 4 entries
print('dim:', x.ndim)       # 1
print('dtype:', x.dtype)

### Geometric view — vectors as arrows

A 2D vector `[a, b]` is an arrow from origin to point `(a, b)`. Higher dimensions work the same — just unvisualizable.

In [ ]:
def plot_vectors(vectors, colors=None, labels=None):
    if colors is None:
        colors = ['C0', 'C1', 'C2', 'C3']
    plt.figure(figsize=(5, 5))
    for i, v in enumerate(vectors):
        plt.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color=colors[i])
        if labels:
            plt.text(v[0]*1.05, v[1]*1.05, labels[i])
    lim = max(abs(v).max() for v in vectors) * 1.3
    plt.xlim(-lim, lim); plt.ylim(-lim, lim)
    plt.axhline(0, color='gray', linewidth=0.5); plt.axvline(0, color='gray', linewidth=0.5)
    plt.grid(True, alpha=0.3); plt.gca().set_aspect('equal')
    plt.show()

u = np.array([3, 2])
v = np.array([-1, 2])
plot_vectors([u, v], labels=['u', 'v'])

## 2. Vector operations

### 2.1 Addition / subtraction — element-wise

```
[a, b] + [c, d] = [a+c, b+d]
```

Shapes must match. Geometrically: head-to-tail.

**Where it shows up in ML:** gradient descent update `w ← w − η · gradient`. Every training step is a vector subtraction.

In [ ]:
u = np.array([3, 2])
v = np.array([-1, 2])

print('u + v =', u + v)
print('u - v =', u - v)

plot_vectors([u, v, u + v], labels=['u', 'v', 'u+v'], colors=['C0', 'C1', 'C2'])

### 2.2 Scalar multiplication — scale every component

```
3 · [a, b] = [3a, 3b]
```

**Where it shows up in ML:** learning rate `η` multiplying the gradient. Also normalizing a vector to unit length.

In [ ]:
u = np.array([3, 2])
print('2 * u =', 2 * u)
print('-0.5 * u =', -0.5 * u)

plot_vectors([u, 2*u, -0.5*u], labels=['u', '2u', '-0.5u'])

## 3. The dot product — the most important operation

```
x · w = x₁w₁ + x₂w₂ + ... + xₙwₙ    →  one number (scalar)
```

Two equivalent readings:

- **Algebraic:** sum of element-wise products.
- **Geometric:** `|x| · |w| · cos(θ)` — measures how aligned two vectors are.

| Angle θ | cos(θ) | Dot product | Meaning |
|---------|--------|-------------|---------|
| 0° | 1 | `|x|·|w|` (max +) | same direction |
| 90° | 0 | 0 | perpendicular / unrelated |
| 180° | -1 | `-|x|·|w|` (max −) | opposite direction |

**Where it shows up in ML:** every linear model computes `score = w · x + b`. SGDClassifier, linear regression, logistic regression, SVMs — all the same core operation. One layer of a neural network is a batch of dot products.

In [ ]:
x = np.array([1, 2, 3])
w = np.array([4, 5, 6])

# Three equivalent ways to write it
print('x @ w      =', x @ w)
print('np.dot     =', np.dot(x, w))
print('manual sum =', (x * w).sum())

In [ ]:
# Geometric intuition: dot product measures alignment
def cos_angle(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

a = np.array([1, 0])

for b, label in [(np.array([1, 0]),  'same dir'),
                 (np.array([1, 1]),  '45°'),
                 (np.array([0, 1]),  'perpendicular'),
                 (np.array([-1, 0]), 'opposite')]:
    print(f'{label:15s}  dot = {a @ b:>5.2f}   cos = {cos_angle(a, b):>5.2f}')

### Why this is *the* operation for classification

Think of `w` as a "direction in feature space that points toward class 1." The dot product asks:

> How much does input `x` point along `w`?

- Large positive → strong yes (predict class 1)
- Near zero → on the fence
- Large negative → strong no (predict class 0)

Training's whole job is to *find* the right `w`.

## 3.5 Geometric intuition — why this matters in XY space

ML operates in high-dim space (MNIST: 784-D), but the *same* geometric rules hold in 2D where you can see them. Build the intuition in 2D — then trust that it generalizes.

Three pictures worth owning:
1. Dot product as a **projection** (shadow).
2. Weight vector `w` defines a **hyperplane** — the decision boundary.
3. Gradient descent as **walking downhill** on a loss surface.

### 3.5.1 Dot product = projection

`x · w` has a geometric meaning: it's the **length of x's shadow on the direction of w**, scaled by `|w|`.

```
x · w = |w| · (length of x projected onto w)
```

So when a classifier computes `w · x`:
- It asks: **how far along direction `w` does input `x` lie?**
- Points on the `w` side → positive score → class 1.
- Points on the opposite side → negative score → class 0.
- Points perpendicular to `w` → score 0 → right on the boundary.

`w` is a **direction that points toward class 1** in feature space. Training finds this direction.

### Geometric picture — why L1 produces sparse weights

The **unit ball** is the set of all vectors with norm ≤ 1. Its shape depends on which norm you use:

- **L2 unit ball** = a **circle** (smooth, round)
- **L1 unit ball** = a **diamond** (sharp corners on the axes)
- **L∞ unit ball** = a **square**

When training with regularization, you're minimizing loss *while keeping* `‖w‖` small. Geometrically: the loss contours expand outward from the best-fit `w`, and you stop at the first point they touch the norm's unit ball.

- **Ridge (L2)**: first contact is usually somewhere on the smooth circle — weights shrink toward zero but rarely hit it exactly.
- **Lasso (L1)**: first contact tends to land on a **corner** of the diamond, where one or more coordinates are exactly zero → **feature selection falls out of the geometry**.

That's why Lasso zeroes out weights and Ridge doesn't. It's a shape fact.

In [ ]:
theta = np.linspace(0, 2*np.pi, 400)
xs = np.cos(theta); ys = np.sin(theta)

# Build unit balls in L1, L2, L∞
l2 = np.stack([xs, ys], axis=1)

t = np.linspace(-1, 1, 100)
l1 = np.concatenate([
    np.stack([t, 1 - np.abs(t)], axis=1),
    np.stack([t, -(1 - np.abs(t))], axis=1)
])

linf = np.array([[1,1], [-1,1], [-1,-1], [1,-1], [1,1]])

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, data, title in zip(axes,
                           [l1, l2, linf],
                           ['L1 unit ball (diamond)', 'L2 unit ball (circle)', 'L∞ unit ball (square)']):
    ax.plot(data[:, 0], data[:, 1], 'C0')
    ax.fill(data[:, 0], data[:, 1], alpha=0.2)
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.axhline(0, color='gray', lw=0.3); ax.axvline(0, color='gray', lw=0.3)
    ax.grid(alpha=0.3); ax.set_aspect('equal'); ax.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
w = np.array([2.0, 1.0])
x = np.array([1.5, 2.5])

# Scalar projection of x onto w
proj_scalar = (x @ w) / np.linalg.norm(w)           # length of shadow
proj_vector = proj_scalar * (w / np.linalg.norm(w)) # shadow as a vector

plt.figure(figsize=(6, 6))
plt.quiver(0, 0, *w, angles='xy', scale_units='xy', scale=1, color='C0', label='w (direction)')
plt.quiver(0, 0, *x, angles='xy', scale_units='xy', scale=1, color='C1', label='x (input)')
plt.quiver(0, 0, *proj_vector, angles='xy', scale_units='xy', scale=1, color='C2', label='projection of x on w')
plt.plot([x[0], proj_vector[0]], [x[1], proj_vector[1]], 'k--', alpha=0.5)
plt.xlim(-1, 4); plt.ylim(-1, 4)
plt.axhline(0, color='gray', lw=0.5); plt.axvline(0, color='gray', lw=0.5)
plt.grid(alpha=0.3); plt.gca().set_aspect('equal'); plt.legend()
plt.title(f'x · w = {x @ w:.2f}   (projection length = {proj_scalar:.2f})')
plt.show()

### 3.5.2 The hyperplane — where the decision lives

A linear classifier's decision rule is:

```
w · x + b  >  0   →  predict class 1
w · x + b  <  0   →  predict class 0
w · x + b  =  0   →  on the boundary
```

The set of points where `w · x + b = 0` is called a **hyperplane**:
- In 2D: a line
- In 3D: a plane
- In n-D: an (n−1)-dim flat surface

For MNIST's "is it a 5?", the decision boundary is a **783-dimensional hyperplane** living inside 784-D space. Unvisualizable — but geometrically identical to the 2D picture below.

Key property: **`w` is perpendicular to the hyperplane**. Training = rotating and shifting this boundary until it separates the classes well.

### Geometric picture — a matrix *is* a transformation

`A · x` takes the point `x` and moves it somewhere else. Apply `A` to every point in a grid and you see the transformation as a whole: scale, rotate, shear, or any combination.

This is exactly what happens inside a neural network layer: `layer(x) = W · x + b`. The weight matrix `W` warps feature space; training learns a warp that makes classes separable.

**Matrix multiplication = composing transformations.** `(A · B) · x` means \"first apply `B`, then apply `A`.\" Stacking layers in a network is stacking matrix transformations.

In [ ]:
def plot_transform(A, title):
    # Original unit grid + a shape (a square) with a colored arrow
    grid = np.stack(np.meshgrid(np.linspace(-2, 2, 9), np.linspace(-2, 2, 9)), axis=-1).reshape(-1, 2)
    shape = np.array([[1, 0.5], [1, -0.5], [-1, -0.5], [-1, 0.5], [1, 0.5]])

    # Apply A to every point
    grid_t  = (A @ grid.T).T
    shape_t = (A @ shape.T).T

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, g, s, label in [(axes[0], grid, shape, 'before'),
                            (axes[1], grid_t, shape_t, 'after A·x')]:
        ax.scatter(g[:, 0], g[:, 1], s=8, c='lightgray')
        ax.plot(s[:, 0], s[:, 1], 'C0-', lw=2)
        ax.fill(s[:, 0], s[:, 1], alpha=0.3)
        ax.axhline(0, color='gray', lw=0.3); ax.axvline(0, color='gray', lw=0.3)
        ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
        ax.grid(alpha=0.3); ax.set_aspect('equal'); ax.set_title(label)
    fig.suptitle(title); plt.tight_layout(); plt.show()

# Three canonical transformations
plot_transform(np.array([[2, 0], [0, 1]]),               'Scaling (stretch x by 2)')
plot_transform(np.array([[np.cos(0.6), -np.sin(0.6)],
                         [np.sin(0.6),  np.cos(0.6)]]),   'Rotation (~34°)')
plot_transform(np.array([[1, 1], [0, 1]]),               'Shear')

In [ ]:
rng = np.random.default_rng(0)

# Two clusters of 2D points — 'not 5' on the left, '5' on the right
class0 = rng.normal(loc=[-1.5, -1.0], scale=0.6, size=(30, 2))
class1 = rng.normal(loc=[ 1.5,  1.0], scale=0.6, size=(30, 2))

# A chosen separator: w · x + b = 0
w = np.array([1.0, 0.8])
b = -0.3

# Boundary line: w[0]*x + w[1]*y + b = 0  →  y = -(w[0]*x + b) / w[1]
xs = np.linspace(-3.5, 3.5, 100)
ys = -(w[0] * xs + b) / w[1]

plt.figure(figsize=(7, 6))
plt.scatter(*class0.T, color='C0', label='class 0 (not 5)')
plt.scatter(*class1.T, color='C3', label='class 1 (is 5)')
plt.plot(xs, ys, 'k-', label='decision boundary w·x + b = 0')
plt.quiver(0, -b/w[1], *w, angles='xy', scale_units='xy', scale=1,
           color='green', label='w (perpendicular to boundary)')
plt.xlim(-3.5, 3.5); plt.ylim(-3.5, 3.5)
plt.axhline(0, color='gray', lw=0.3); plt.axvline(0, color='gray', lw=0.3)
plt.grid(alpha=0.3); plt.gca().set_aspect('equal'); plt.legend()
plt.title('Linear classifier in 2D — exactly what SGDClassifier learns')
plt.show()

### 3.5.3 Gradient descent — walking downhill

Training asks: **which `w` minimizes the loss?** The loss is a function from weight-space → one number:

```
L(w) = (how wrong the model is with weights w)
```

You can picture `L` as a landscape over weight-space:
- Each point `(w₁, w₂)` on the floor is one possible weight setting.
- The height at that point is the loss.
- **Low = good**, **high = bad**.

The **gradient** `∇L(w)` is a vector pointing in the direction of *steepest increase*. So gradient descent does:

```
w ← w − η · ∇L(w)       # step against the gradient = downhill
```

For a **convex** loss (bowl-shaped, like MSE and hinge loss), any downhill path reaches the global minimum. For **non-convex** losses (neural networks), you can get stuck in local minima — but in practice SGD finds good ones.

In [ ]:
# Toy convex loss: L(w1, w2) = w1² + 3*w2²  (an elongated bowl)
def loss(w):
    return w[0]**2 + 3 * w[1]**2

def grad(w):
    return np.array([2 * w[0], 6 * w[1]])

# Run gradient descent from a starting point
w = np.array([3.5, 2.5])
eta = 0.1
path = [w.copy()]
for _ in range(20):
    w = w - eta * grad(w)
    path.append(w.copy())
path = np.array(path)

# Contour map + descent path
w1 = np.linspace(-4, 4, 100)
w2 = np.linspace(-4, 4, 100)
W1, W2 = np.meshgrid(w1, w2)
Z = W1**2 + 3 * W2**2

plt.figure(figsize=(7, 6))
plt.contour(W1, W2, Z, levels=20, cmap='viridis', alpha=0.6)
plt.plot(path[:, 0], path[:, 1], 'ro-', markersize=5, label='gradient descent path')
plt.plot(0, 0, 'g*', markersize=20, label='minimum (best w)')
plt.xlabel('w₁'); plt.ylabel('w₂')
plt.title('Gradient descent on a convex loss — each red dot is one update step')
plt.grid(alpha=0.3); plt.legend(); plt.gca().set_aspect('equal')
plt.show()

print(f'Final w = {path[-1].round(3)},  final loss = {loss(path[-1]):.4f}')

### How this generalizes beyond 2D

Everything you just saw in 2D holds in 784-D — you just can't draw it:

| 2D picture | MNIST reality |
|-----------|---------------|
| Arrow in the plane | 784-component vector |
| Line separating dots | 783-D hyperplane separating images |
| Bowl-shaped loss surface | Bowl in 784-D weight space |
| Red dot walking downhill | 785 numbers updating each SGD step |

The geometry doesn't change. Only the axis count does.

## 4. Norms — how "big" is a vector

```
‖x‖₂ = √(x₁² + x₂² + ... + xₙ²)    ← L2 norm (Euclidean, default)
‖x‖₁ = |x₁| + |x₂| + ... + |xₙ|    ← L1 norm (Manhattan)
```

**Where it shows up in ML:** regularization (Ch. 4). Penalize `‖w‖` to prevent overfitting.
- L2 penalty → **Ridge regression** (shrinks all weights smoothly)
- L1 penalty → **Lasso regression** (shrinks some weights to exactly zero — feature selection)

In [ ]:
x = np.array([3, 4])

print('L2 norm:', np.linalg.norm(x))         # √(9+16) = 5
print('L1 norm:', np.linalg.norm(x, ord=1))  # |3|+|4| = 7
print('L∞ norm:', np.linalg.norm(x, ord=np.inf))  # max(|3|, |4|) = 4

## 5. Matrices

A matrix is a rectangular grid of numbers. Shape `(m, n)` = m rows, n columns.

In ML, the **training set** is a matrix:
- Rows = examples (60,000 for MNIST train)
- Columns = features (784 for MNIST)
- Shape: `X ∈ ℝ⁶⁰⁰⁰⁰ˣ⁷⁸⁴`

Each row `X[i]` is one input vector, paired with label `y[i]`.

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

print('A =')
print(A)
print('shape:', A.shape)   # (2, 3) — 2 rows, 3 cols
print('ndim :', A.ndim)    # 2
print('row 0:', A[0])      # [1 2 3]
print('col 1:', A[:, 1])   # [2 5]

### Special matrices

In [ ]:
print('zeros:\n', np.zeros((2, 3)))
print('ones:\n',  np.ones((2, 3)))
print('identity:\n', np.eye(3))    # 1s on diagonal, 0s elsewhere
print('random:\n', np.random.randn(2, 3))   # normal distribution

## 6. Matrix operations

### 6.1 Matrix × vector — a transformation

```
A · x = y    where A is (m, n), x is (n,), y is (m,)
```

`A` transforms an n-dim vector into an m-dim vector. Each output component `y[i]` is the dot product of row `i` of `A` with `x`.

**Where it shows up in ML:**
- One layer of a neural network: `output = W · input + b`
- Scoring a whole batch at once: `X · w` gives one score per row (see §8)

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])             # (2, 3)
x = np.array([1, 0, -1])              # (3,)

y = A @ x
print('A @ x =', y)
print('shape:', y.shape)              # (2,)

# Verify: each output = dot product of row with x
print('row 0 · x =', A[0] @ x)
print('row 1 · x =', A[1] @ x)

### 6.2 Matrix × matrix

```
A · B = C    where A is (m, n), B is (n, p), C is (m, p)
```

**Inner dimensions must match** (`n = n`). The `(i, j)` entry of `C` is the dot product of row `i` of `A` with column `j` of `B`.

Mnemonic: `(m, n) @ (n, p) = (m, p)` — inner n's cancel.

## 8.5 End-to-end payoff — training a model with *only* linear algebra

For ordinary least squares (linear regression with MSE loss), there's a **closed-form solution** — no iteration required. The best weights `θ = [w; b]` are:

```
θ = (Xᵀ X)⁻¹ Xᵀ y       ← the normal equation
```

Every symbol on the right is linear algebra:
- `X` — data matrix `(m examples, n features + 1 for bias)`
- `Xᵀ` — transpose
- `Xᵀ X` — `(n, n)` Gram matrix
- `(·)⁻¹` — matrix inverse
- `Xᵀ y` — `(n,)` vector

One formula. No gradient descent, no learning rate — just shapes that must line up. Let's run it on synthetic data.

In [ ]:
rng = np.random.default_rng(0)

# Generate data from true weights: y = 2*x1 + 0.5*x2 + 1 + noise
m = 200
X_features = rng.normal(size=(m, 2))
w_true = np.array([2.0, 0.5])
b_true = 1.0
y = X_features @ w_true + b_true + rng.normal(scale=0.3, size=m)

# Add a column of 1s to fold bias into the weight vector
X_bias = np.hstack([X_features, np.ones((m, 1))])          # (200, 3)

# Normal equation:  θ = (Xᵀ X)⁻¹ Xᵀ y
theta = np.linalg.inv(X_bias.T @ X_bias) @ X_bias.T @ y    # (3,)

print('Estimated w:  ', theta[:2].round(3),  '  (true:', w_true.tolist(), ')')
print('Estimated b:  ', theta[2].round(3),   '  (true:', b_true, ')')

# Predict on new data — one matrix multiply
y_pred = X_bias @ theta
mse = ((y_pred - y) ** 2).mean()
print(f'MSE on training data: {mse:.4f}')

## 11. Linear algebra ⇄ ML concept map

| Linear algebra | ML equivalent |
|----------------|---------------|
| Vector `x ∈ ℝⁿ` | One training example with `n` features |
| Matrix `X ∈ ℝᵐˣⁿ` | Entire training set — `m` examples, `n` features |
| Weight vector `w` | Learned parameters (same shape as one input) |
| Dot product `w · x` | The score of a linear classifier/regressor |
| `w · x = 0` hyperplane | Decision boundary of every linear model |
| Matrix × vector `X · w` | Score every example in a batch at once |
| Matrix × matrix | Composing transformations (stacked NN layers) |
| Transpose `Xᵀ` | Appears in the normal equation and in backprop |
| L2 norm `‖w‖₂` | Ridge regularization penalty |
| L1 norm `‖w‖₁` | Lasso regularization penalty (induces sparsity) |
| Vector subtraction `w − η·g` | One gradient-descent update step |
| `(XᵀX)⁻¹Xᵀy` | Closed-form linear regression (normal equation) |
| Broadcasting | Adding a bias vector to every row of a batch |

**What just happened — calculus-free training.** No loop, no learning rate, no gradient. One matrix inverse + three matrix multiplies and we recovered the true parameters. This is the full training algorithm for ordinary least squares, written entirely in linear algebra. (Scikit-learn's `LinearRegression` does essentially this; `SGDRegressor` uses the iterative version from Ch. 4.)

When does this close-form shortcut stop working? When `XᵀX` is too big to invert (Ch. 4's motivation for gradient descent) or when the loss isn't MSE (no closed form exists).

In [ ]:
A = np.array([[1, 2], [3, 4]])           # (2, 2)
B = np.array([[5, 6, 7], [8, 9, 10]])    # (2, 3)

C = A @ B
print('C =\n', C)
print('shape:', C.shape)                  # (2, 3)

# Not commutative: A @ B != B @ A (often not even same shape)
try:
    B @ A
except ValueError as e:
    print('B @ A fails:', e)

### 6.3 Transpose — flip rows and columns

```
A is (m, n)  →  Aᵀ is (n, m)
```

**Where it shows up in ML:** formulas like `Xᵀ · X` (normal equation, Ch. 4), `wᵀ · x` (just another way to write dot product).

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])   # (2, 3)

print('A =\n', A)
print('A.T =\n', A.T)       # (3, 2)
print('A.T.shape:', A.T.shape)

## 7. Shapes and broadcasting

Most ML bugs are shape bugs. Memorize these:

| Operation | Requirement |
|-----------|-------------|
| `a + b`, `a * b` (element-wise) | same shape, or broadcastable |
| `a @ b` (vector dot) | same length |
| `A @ B` (matmul) | inner dims match: `(m, n) @ (n, p)` |

### Broadcasting

NumPy lets you combine arrays of compatible shapes without an explicit loop. Rule: align shapes from the **right**; each dim must be equal or 1.

In [ ]:
X = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])           # (3, 3)
bias = np.array([10, 20, 30])        # (3,)

# Broadcasting: bias is added to every row
print('X + bias =\n', X + bias)

# Scalar broadcasting
print('X * 2 =\n', X * 2)

In [ ]:
# Habit: print .shape whenever something breaks
a = np.random.randn(60000, 784)     # like MNIST X_train
w = np.random.randn(784)
b = 0.5

scores = a @ w + b                   # (60000, 784) @ (784,) + scalar = (60000,)
print('scores.shape:', scores.shape)

### Common shape gotchas

In [ ]:
# (n,) vs (n, 1) vs (1, n) — all 'look' like a list but behave differently
v  = np.array([1, 2, 3])          # (3,)    — 1D
vc = v.reshape(-1, 1)             # (3, 1)  — column
vr = v.reshape(1, -1)             # (1, 3)  — row

print('v  shape:', v.shape)
print('vc shape:', vc.shape)
print('vr shape:', vr.shape)

# vc @ vr gives (3, 3) outer product; vr @ vc gives (1, 1) scalar-ish
print('vc @ vr =\n', vc @ vr)
print('vr @ vc =', vr @ vc)

## 8. Putting it together — how linear algebra drives ML

For a linear classifier / regressor on one example:

```
Input:       x ∈ ℝⁿ             ← one example, n features
Weights:     w ∈ ℝⁿ             ← learned, same shape as x
Bias:        b ∈ ℝ              ← one number
Score:       s = w · x + b      ← dot product + scalar
Prediction:  ŷ = f(s)           ← f = sign / sigmoid / softmax / identity
```

For a whole batch of `m` examples:

```
X ∈ ℝᵐˣⁿ,    scores = X · w + b ∈ ℝᵐ
```

Let's build that end-to-end with MNIST-shaped fake data.

In [ ]:
rng = np.random.default_rng(42)

m, n = 60000, 784                   # examples, features

X = rng.normal(size=(m, n))         # fake training data
w = rng.normal(size=n) * 0.01       # small random weights
b = 0.0                             # bias

# Score every example at once — one line, no loop
scores = X @ w + b                  # shape (60000,)

# Binary prediction: positive score → class 1
y_pred = (scores > 0).astype(int)

print('X.shape     =', X.shape)
print('w.shape     =', w.shape)
print('scores.shape=', scores.shape)
print('y_pred.shape=', y_pred.shape)
print('first 10 scores:', scores[:10].round(3))
print('first 10 preds :', y_pred[:10])

### A single gradient-descent step in pure linear algebra

Even without knowing the exact loss, the update shape is always:

```
w ← w − η · gradient       # vector subtraction, scalar multiply
```

In [ ]:
eta = 0.001                                    # learning rate
gradient = rng.normal(size=n)                  # pretend gradient

w_new = w - eta * gradient                     # the update

print('||w||       =', np.linalg.norm(w).round(4))
print('||w_new||   =', np.linalg.norm(w_new).round(4))
print('||update||  =', np.linalg.norm(eta * gradient).round(4))

## 9. Deferred — come back when a chapter needs it

| Topic | When it becomes relevant |
|-------|--------------------------|
| Determinant | Rare in HOML. Safe to skip. |
| Matrix inverse | Normal equation, Ch. 4. Learn then. |
| Eigenvalues / eigenvectors | PCA, Ch. 8. Learn then. |
| SVD | Also PCA, Ch. 8. Learn then. |
| QR / LU / Cholesky decompositions | Numerical methods — optional. |

## 10. NumPy cheat sheet

In [ ]:
# --- Create ---
x = np.array([1, 2, 3])           # vector   (3,)
A = np.array([[1, 2], [3, 4]])    # matrix   (2, 2)
np.zeros((3, 4));  np.ones((2,)); np.eye(3); np.random.randn(3, 2)

# --- Shape / inspect ---
x.shape; x.ndim; x.dtype; x.size
x.reshape(-1, 1);  x[np.newaxis, :]    # add axis

# --- Element-wise ---
x + 1;  x * 2;  x ** 2;  np.exp(x);  np.log(x + 1)

# --- Linear algebra ---
x @ x                                # dot product
A @ x                                # matrix × vector
A @ A                                # matrix × matrix
A.T                                  # transpose
np.linalg.norm(x)                    # L2 norm
np.linalg.norm(x, ord=1)             # L1 norm

# --- Reductions ---
x.sum(); x.mean(); x.std(); x.max()
A.sum(axis=0)    # sum down columns → (ncols,)
A.sum(axis=1)    # sum across rows  → (nrows,)

print('cheat sheet cell ran — nothing printed but everything is valid')

## Minimum bar before moving on

You should be able to:

- [ ] Write `w · x + b` and state every shape for MNIST (`x: (784,)`, `w: (784,)`, `b: scalar`, output: scalar).
- [ ] Explain why `X @ w` gives scores for the whole batch at once, and what its shape is.
- [ ] Predict whether `(m, n) @ (p, q)` works, and if so what shape comes out.
- [ ] Know that gradient descent updates are vector subtractions scaled by `η`.
- [ ] Read `‖w‖₂` and `‖w‖₁` and know which is Ridge vs Lasso.

That's enough for Ch. 3 – 7. The rest compounds as the book demands it.